In [2]:
# get the size of a file

import os

def get_file_size(file_path):
    return os.path.getsize(file_path)

os.path.getsize('pyproject.toml')

352

In [168]:
from woc.local import WocMapsLocal

woc = WocMapsLocal()

In [200]:
from typing import Tuple 
import os
import hashlib

def sample_md5(
        file_path: str,
        skip = 0,
        size = None
    ) -> str:
    """
    Divide the file into chunks and calculate the MD5 digest of the first 128 bytes of each chunk.
    
    :param file_path: The path to the file.
    :param skip: The number of bytes to skip from the beginning of the file. Defaults to 0.
    :param size: The number of bytes to for hashing. If None, the entire file (minus the skip) is considered.
    :return: A tuple containing the size of the considered portion and the 16-character MD5 digest.
    """
    
    fsize = os.path.getsize(file_path)
    
    if size is None:
        size = fsize - skip    
    assert skip + size <= fsize, f"supplied size {size}B > file size {os.path.getsize(file_path)}B"

    dig = hashlib.md5()
    
    # hash all bytes if file is small
    if size <= 4096:  # typical block size of ext4
        with open(file_path, 'rb') as f:
            f.seek(skip)
            dig.update(f.read(size))
            return size, dig.hexdigest()[:16]
    
    # A heuristic to find the optimal chunk size
    # Number of chunks is between 2 and 8, chunk size must be a power of 2
    chunk_size = 2 ** ((size // size.bit_length()).bit_length() + 2)
    num_chunks = (size - 256) // chunk_size  # don't hash the same bytes twice
    
    with open(file_path, 'rb') as f:
        f.seek(skip)
        dig.update(f.read(128))
        for _ in range(num_chunks):
            f.seek(chunk_size - 128, os.SEEK_CUR)
            print(f.tell())
            dig.update(f.read(128))
        f.seek(skip + size - 128)
        dig.update(f.read(128))
        
    return size, dig.hexdigest()[:16]

In [201]:
sample_md5('/woc/All.blobs/blob_0.bin', size=688438563942)

137438953472
274877906944
412316860416
549755813888
687194767360


(688438563942, 'd6e2baa102edcec7')

In [177]:
from dataclasses import dataclass, asdict

@dataclass
class WocFile:
    path: str
    digest: str
    size: int
    mtime: int

In [190]:
digests = {}

_file_lists = []

for m in woc.config["maps"].values():
    for mm in m:
        for s in mm["shards"]:
            _file_lists.append(s)
        for l in mm["larges"].values():
            _file_lists.append(l)
for m in woc.config["objects"].values():
    _file_lists.extend(m["shards"])
        
_file_lists

['/woc/basemaps/b2faFullV.0.tch',
 '/woc/basemaps/b2faFullV.1.tch',
 '/woc/basemaps/b2faFullV.2.tch',
 '/woc/basemaps/b2faFullV.3.tch',
 '/woc/basemaps/b2faFullV.4.tch',
 '/woc/basemaps/b2faFullV.5.tch',
 '/woc/basemaps/b2faFullV.6.tch',
 '/woc/basemaps/b2faFullV.7.tch',
 '/woc/basemaps/b2faFullV.8.tch',
 '/woc/basemaps/b2faFullV.9.tch',
 '/woc/basemaps/b2faFullV.10.tch',
 '/woc/basemaps/b2faFullV.11.tch',
 '/woc/basemaps/b2faFullV.12.tch',
 '/woc/basemaps/b2faFullV.13.tch',
 '/woc/basemaps/b2faFullV.14.tch',
 '/woc/basemaps/b2faFullV.15.tch',
 '/woc/basemaps/b2faFullV.16.tch',
 '/woc/basemaps/b2faFullV.17.tch',
 '/woc/basemaps/b2faFullV.18.tch',
 '/woc/basemaps/b2faFullV.19.tch',
 '/woc/basemaps/b2faFullV.20.tch',
 '/woc/basemaps/b2faFullV.21.tch',
 '/woc/basemaps/b2faFullV.22.tch',
 '/woc/basemaps/b2faFullV.23.tch',
 '/woc/basemaps/b2faFullV.24.tch',
 '/woc/basemaps/b2faFullV.25.tch',
 '/woc/basemaps/b2faFullV.26.tch',
 '/woc/basemaps/b2faFullV.27.tch',
 '/woc/basemaps/b2faFullV.28.t

In [191]:
_file_lists[-10:]

['/woc/All.sha1o/sha1.commit_118.tch',
 '/woc/All.sha1o/sha1.commit_119.tch',
 '/woc/All.sha1o/sha1.commit_120.tch',
 '/woc/All.sha1o/sha1.commit_121.tch',
 '/woc/All.sha1o/sha1.commit_122.tch',
 '/woc/All.sha1o/sha1.commit_123.tch',
 '/woc/All.sha1o/sha1.commit_124.tch',
 '/woc/All.sha1o/sha1.commit_125.tch',
 '/woc/All.sha1o/sha1.commit_126.tch',
 '/woc/All.sha1o/sha1.commit_127.tch']

In [193]:
from tqdm import tqdm

for f in tqdm(_file_lists):
    if not f:
        continue
    _fname = os.path.basename(f)
    size, digest = sample_md5(f)
    digests[_fname] = asdict(WocFile(f, digest, size, os.path.getmtime(f)))
    
digests

100%|██████████| 3857/3857 [01:49<00:00, 35.16it/s]  


{'b2faFullV.0.tch': {'path': '/woc/basemaps/b2faFullV.0.tch',
  'digest': '13f573f8527713ea',
  'size': 66194145056,
  'mtime': 1710207932.56853},
 'b2faFullV.1.tch': {'path': '/woc/basemaps/b2faFullV.1.tch',
  'digest': 'fdcb41fe04ed0b8c',
  'size': 66191648112,
  'mtime': 1710252815.118011},
 'b2faFullV.2.tch': {'path': '/woc/basemaps/b2faFullV.2.tch',
  'digest': '8e75d6442ad805ff',
  'size': 66201770960,
  'mtime': 1710253641.562702},
 'b2faFullV.3.tch': {'path': '/woc/basemaps/b2faFullV.3.tch',
  'digest': 'c5684192f5b35216',
  'size': 66201470176,
  'mtime': 1710254593.012034},
 'b2faFullV.4.tch': {'path': '/woc/basemaps/b2faFullV.4.tch',
  'digest': 'c86dd7028e75b35b',
  'size': 66201163840,
  'mtime': 1710253355.9010024},
 'b2faFullV.5.tch': {'path': '/woc/basemaps/b2faFullV.5.tch',
  'digest': '99aa81043e1dabec',
  'size': 66199145520,
  'mtime': 1710268639.6042805},
 'b2faFullV.6.tch': {'path': '/woc/basemaps/b2faFullV.6.tch',
  'digest': '55c94f2ac594f193',
  'size': 6620028

In [194]:
import json

with open('woc.digests.json', 'w') as f:
    json.dump(digests, f)

In [160]:
size = 4096003423
chunk_size = 2 ** ((size // size.bit_length()).bit_length() + 2)
chunk_size, size, (size - 128) // chunk_size

(536870912, 4096003423, 7)

In [129]:
for i in range(0, 48):
    size = 2 ** i
    chunk_size = 2 ** ((size // size.bit_length()).bit_length() + 2)
    print(i, chunk_size, size, size // chunk_size)

0 8 1 0
1 8 2 0
2 8 4 0
3 16 8 0
4 16 16 1
5 32 32 1
6 64 64 1
7 128 128 1
8 128 256 2
9 256 512 2
10 512 1024 2
11 1024 2048 2
12 2048 4096 2
13 4096 8192 2
14 8192 16384 2
15 16384 32768 2
16 16384 65536 4
17 32768 131072 4
18 65536 262144 4
19 131072 524288 4
20 262144 1048576 4
21 524288 2097152 4
22 1048576 4194304 4
23 2097152 8388608 4
24 4194304 16777216 4
25 8388608 33554432 4
26 16777216 67108864 4
27 33554432 134217728 4
28 67108864 268435456 4
29 134217728 536870912 4
30 268435456 1073741824 4
31 536870912 2147483648 4
32 536870912 4294967296 8
33 1073741824 8589934592 8
34 2147483648 17179869184 8
35 4294967296 34359738368 8
36 8589934592 68719476736 8
37 17179869184 137438953472 8
38 34359738368 274877906944 8
39 68719476736 549755813888 8
40 137438953472 1099511627776 8
41 274877906944 2199023255552 8
42 549755813888 4398046511104 8
43 1099511627776 8796093022208 8
44 2199023255552 17592186044416 8
45 4398046511104 35184372088832 8
46 8796093022208 70368744177664 8
47 17

In [142]:
size = 12 * 1024 * 1024 * 1024  # 64 GB
chunk_size = 4096 * (size // size.bit_length())
size // chunk_size

0

In [44]:
size = 1 * 1024 * 1024 * 1024  # 64 GB

2 ** (max(1, (size.bit_length()).bit_length()))

32

In [ ]:

offsets = []


In [ ]:
from woc.local import Woc

In [11]:
file_digest('pyproject.toml')

(370, '02cc8826e1ecaebd')

In [5]:
file_digest('pyproject.toml')

(370, '02cc8b57826e1e96caebdfcc')

In [26]:
import hashlib
import os

def _fast_digest(file_path: str, offset = 0, size = None):
    """Calculate the size, head, middle, and tail digests of a file.

    Args:
        file_path (str): The path to the file.
    """

    if size:
        assert offset + size <= os.path.getsize(file_path), f"supplied {size} > local file {os.path.getsize(file_path)}"
    else:
        size = os.path.getsize(file_path) - offset

    if size < 256:
        with open(file_path, 'rb') as f:
            f.seek(offset)
            head = hashlib.md5(f.read(128)).hexdigest()[:8]
            return size, head, head, head

    with open(file_path, 'rb') as f:
        f.seek(offset)
        head = hashlib.md5(f.read(128)).hexdigest()[:8]
        f.seek(offset + size // 2)
        middle = hashlib.md5(f.read(128)).hexdigest()[:8]
        f.seek(offset + size - 128)
        tail = hashlib.md5(f.read(128)).hexdigest()[:8]

    return size, head, middle, tail

_fast_digest('/woc/All.blobs/blob_0.bin')

(688438563942, 'd2db50f7', 'f7805849', 'ee458643')

In [5]:
# calculate the md5 of the first 128-byte of a file

import hashlib

def get_md5(file_path):
    md5 = hashlib.md5()
    with open(file_path, 'rb') as f:
        md5.update(f.read(128))
    return md5.hexdigest()

get_md5('pyproject.toml')

'02cc8b57a0885bc45c4f66508ee672d6'